# 📂 Data Preparation in Climatology Engine

This notebook demonstrates data preparation steps for processing.

**What you will learn:**
- Data quality assessment
- Missing data detection
- Outlier identification
- Data cleaning

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('✅ Libraries loaded.')

In [ ]:
# Load sample data
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))

print(f'📊 Data shape: {station_data.shape}')
station_data.head()

In [ ]:
# Check for missing data
missing = station_data.isnull().sum()
total = len(station_data)

print("📊 Missing data statistics:")
for col in station_data.columns:
    count = missing[col]
    percent = (count / total) * 100
    print(f"   {col}: {count} ({percent:.2f}%)")

if missing.sum() == 0:
    print("\n✅ No missing data.")
else:
    print("\n⚠️ Missing data detected.")

In [ ]:
# Outlier detection using IQR method
data = station_data.values

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(['tmin', 'tmean', 'tmax']):
    axes[i].boxplot(data[:, i])
    axes[i].set_title(f'Boxplot - {col}')
    axes[i].set_ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()

In [ ]:
# Outlier detection using Z-score method
def detect_outliers_zscore(data, threshold=3):
    mean = np.mean(data)
    std = np.std(data)
    z_scores = np.abs((data - mean) / std)
    return z_scores > threshold

outliers = {}
for i, col in enumerate(['tmin', 'tmean', 'tmax']):
    mask = detect_outliers_zscore(data[:, i])
    outliers[col] = np.sum(mask)
    print(f"{col}: {outliers[col]} outliers")

total_outliers = sum(outliers.values())
print(f"\n✅ Total outliers: {total_outliers} ({total_outliers/len(data)*100:.2f}%)")

In [ ]:
# Data cleaning function
def clean_data(data, method='remove', threshold=3):
    """
    Clean data by removing or replacing outliers
    """
    cleaned = data.copy()
    for i in range(data.shape[1]):
        mean = np.mean(data[:, i])
        std = np.std(data[:, i])
        z_scores = np.abs((data[:, i] - mean) / std)
        outlier_mask = z_scores > threshold
        
        if method == 'remove':
            cleaned = cleaned[~outlier_mask]
        elif method == 'replace':
            cleaned[outlier_mask, i] = mean
    
    return cleaned

cleaned_data = clean_data(data, method='remove')
print(f"📊 Original data: {data.shape}")
print(f"📊 Cleaned data: {cleaned_data.shape}")

In [ ]:
# Plot comparison before and after cleaning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(['tmean', 'tmax']):
    idx = ['tmin', 'tmean', 'tmax'].index(col)
    axes[i].boxplot([data[:, idx], cleaned_data[:, idx]], labels=['Before', 'After'])
    axes[i].set_title(f'{col} - Before vs After Cleaning')
    axes[i].set_ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()

## 📋 Summary

In this notebook you learned:

✅ Checking for missing data
✅ Outlier detection with Boxplot and Z-score
✅ Data cleaning

---

**Next Steps:**
- Notebook 03: Fitting All Distributions